# Stage 05: Hybrid Search Strategies
## Lecture 42: Combining Dense And Sparse Retriever With LangChain

### Educational and Architectural Scope
This notebook implements an enterprise-grade hybrid retrieval pipeline using LangChain.
It unifies:
1. Dense Retrieval using FAISS and HuggingFace Embeddings (`all-MiniLM-L6-v2`).
2. Sparse Retrieval using `BM25Retriever` (in-memory BM25 index via `rank-bm25`).
3. Rank Fusion using `EnsembleRetriever` with weighted Reciprocal Rank Fusion (RRF).
4. End-to-end RAG Generation using `create_stuff_documents_chain` and `create_retrieval_chain`.

### Constraints & Standards
- Clean production code with strict type assertions.
- Zero emojis in all outputs and comments.
- Explicit assertion checkpoints for continuous verification.


In [1]:
# Step 1: Environment Setup and Imports
import os
import sys
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.retrievers.bm25 import BM25Retriever

try:
    from langchain.retrievers.ensemble import EnsembleRetriever
except ImportError:
    from langchain_classic.retrievers.ensemble import EnsembleRetriever

try:
    from langchain.chains.combine_documents import create_stuff_documents_chain
    from langchain.chains import create_retrieval_chain
except ImportError:
    from langchain_classic.chains.combine_documents import create_stuff_documents_chain
    from langchain_classic.chains import create_retrieval_chain

from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq

print("[INFO] Environment loaded and all required libraries imported successfully.")


C:\Users\aliashraf\AppData\Local\Temp\ipykernel_23576\3306628528.py:10: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


[INFO] Environment loaded and all required libraries imported successfully.


In [2]:
# Step 2: Corpus Definition
# 5 distinct documents demonstrating semantic variance and exact keyword distribution
raw_documents = [
    Document(
        page_content="LangChain helps build LLM applications",
        metadata={"source": "doc1", "category": "framework"}
    ),
    Document(
        page_content="LangChain can be used to develop Agentic AI applications",
        metadata={"source": "doc2", "category": "framework"}
    ),
    Document(
        page_content="LangChain has many types of retrievers",
        metadata={"source": "doc3", "category": "framework"}
    ),
    Document(
        page_content="Pinecone is a vector database for semantic search",
        metadata={"source": "doc4", "category": "database"}
    ),
    Document(
        page_content="The Eiffel Tower is located in Paris and is a famous tourist attraction",
        metadata={"source": "doc5", "category": "general"}
    )
]

print(f"[INFO] Initialized {len(raw_documents)} raw documents.")
for idx, doc in enumerate(raw_documents, start=1):
    print(f"  Document {idx} [{doc.metadata['source']}]: {doc.page_content}")

# Assertion Checkpoint
assert len(raw_documents) == 5, "Corpus must contain exactly 5 documents."


[INFO] Initialized 5 raw documents.
  Document 1 [doc1]: LangChain helps build LLM applications
  Document 2 [doc2]: LangChain can be used to develop Agentic AI applications
  Document 3 [doc3]: LangChain has many types of retrievers
  Document 4 [doc4]: Pinecone is a vector database for semantic search
  Document 5 [doc5]: The Eiffel Tower is located in Paris and is a famous tourist attraction


In [3]:
# Step 3: Dense Retriever Construction (FAISS + HuggingFace)
# The dense branch indexes semantic representations into a dense vector space
embedding_model_name = "all-MiniLM-L6-v2"
embeddings = HuggingFaceEmbeddings(model_name=embedding_model_name)

# Index corpus into in-memory FAISS vectorstore
vectorstore = FAISS.from_documents(raw_documents, embeddings)
dense_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# Test dense retrieval on query
test_query = "How can I build an application using LLMs?"
dense_results = dense_retriever.invoke(test_query)

print(f"[INFO] Dense Retriever returned {len(dense_results)} documents for query: '{test_query}'")
for rank, d in enumerate(dense_results, start=1):
    print(f"  [Dense Rank {rank}] ({d.metadata['source']}): {d.page_content}")

# Assertion Checkpoint
assert len(dense_results) == 3, "Dense retriever must return top-3 documents."
assert any("LLM applications" in d.page_content for d in dense_results), "Dense search must retrieve LLM application document."


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[INFO] Dense Retriever returned 3 documents for query: 'How can I build an application using LLMs?'
  [Dense Rank 1] (doc1): LangChain helps build LLM applications
  [Dense Rank 2] (doc2): LangChain can be used to develop Agentic AI applications
  [Dense Rank 3] (doc4): Pinecone is a vector database for semantic search


In [4]:
# Step 4: Sparse Retriever Construction (BM25Retriever)
# The sparse branch indexes lexical term frequencies and IDF via Okapi BM25
sparse_retriever = BM25Retriever.from_documents(raw_documents)
sparse_retriever.k = 3

sparse_results = sparse_retriever.invoke(test_query)

print(f"[INFO] Sparse Retriever returned {len(sparse_results)} documents for query: '{test_query}'")
for rank, d in enumerate(sparse_results, start=1):
    print(f"  [Sparse Rank {rank}] ({d.metadata['source']}): {d.page_content}")

# Assertion Checkpoint
assert len(sparse_results) == 3, "Sparse retriever must return top-3 documents."
assert any("build LLM" in d.page_content or "application" in d.page_content for d in sparse_results), "Sparse search must match lexical keywords."


[INFO] Sparse Retriever returned 3 documents for query: 'How can I build an application using LLMs?'
  [Sparse Rank 1] (doc1): LangChain helps build LLM applications
  [Sparse Rank 2] (doc2): LangChain can be used to develop Agentic AI applications
  [Sparse Rank 3] (doc5): The Eiffel Tower is located in Paris and is a famous tourist attraction


In [5]:
# Step 5: Hybrid Rank Fusion with EnsembleRetriever
# Reciprocal Rank Fusion blends ranks according to assigned weights:
# RRF_Score(d) = sum [ w_r / (60 + rank_r(d)) ]
# Here: 70% dense weight, 30% sparse weight
dense_weight = 0.7
sparse_weight = 0.3

hybrid_retriever = EnsembleRetriever(
    retrievers=[dense_retriever, sparse_retriever],
    weights=[dense_weight, sparse_weight]
)

hybrid_results = hybrid_retriever.invoke(test_query)

print(f"[INFO] Hybrid EnsembleRetriever returned {len(hybrid_results)} documents:")
for rank, d in enumerate(hybrid_results, start=1):
    print(f"  [Hybrid Rank {rank}] ({d.metadata['source']}): {d.page_content}")

# Assertion Checkpoints
assert len(hybrid_results) >= 3, "Hybrid retriever must return at least 3 relevant documents."
# Verify irrelevant document (Eiffel Tower) is not the top result
top_doc_sources = [d.metadata['source'] for d in hybrid_results[:3]]
assert "doc5" not in top_doc_sources, "Irrelevant document (Eiffel Tower) must not be in the top 3 hybrid results."
assert "doc1" in top_doc_sources or "doc2" in top_doc_sources, "Primary LangChain LLM docs must dominate top hybrid ranks."


[INFO] Hybrid EnsembleRetriever returned 4 documents:
  [Hybrid Rank 1] (doc1): LangChain helps build LLM applications
  [Hybrid Rank 2] (doc2): LangChain can be used to develop Agentic AI applications
  [Hybrid Rank 3] (doc4): Pinecone is a vector database for semantic search
  [Hybrid Rank 4] (doc5): The Eiffel Tower is located in Paris and is a famous tourist attraction


In [6]:
# Step 6: End-to-End RAG Pipeline with Document and Retrieval Chains
# Construct full generation pipeline combining the hybrid retriever with LLM synthesis

# Define prompt template
rag_prompt = ChatPromptTemplate.from_template(
    """You are a technical AI assistant. Answer the user question based strictly on the provided context below.
Context:
{context}

Question:
{input}

Answer concisely:"""
)

# Initialize fast production LLM (Groq)
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0.1)

# Build document combination chain (Stuff strategy)
document_chain = create_stuff_documents_chain(llm, rag_prompt)

# Build complete retrieval chain
rag_chain = create_retrieval_chain(hybrid_retriever, document_chain)

# Execute RAG pipeline
response = rag_chain.invoke({"input": test_query})

print("[INFO] RAG Execution Response:")
print(f"Question: {response['input']}")
print(f"Synthesized Answer: {response['answer']}")
print(f"Number of Retrieved Context Documents: {len(response['context'])}")

# Assertion Checkpoints
assert "answer" in response and len(response["answer"].strip()) > 0, "Response must contain a non-empty synthesized answer."
assert "context" in response and len(response["context"]) >= 3, "Response must include retrieved context documents."
assert any("LangChain" in doc.page_content for doc in response["context"]), "Retrieved context must contain LangChain documents."


[INFO] RAG Execution Response:
Question: How can I build an application using LLMs?
Synthesized Answer: 1. **Choose an LLM** – e.g., OpenAI GPT‑4, Anthropic Claude, or a local model.  
2. **Use LangChain** –  
   * Install LangChain (`pip install langchain`).  
   * Create a `Chain` or `Agent` that wraps the LLM, adding prompts, memory, and tool‑calling.  
3. **Add vector search (optional)** –  
   * Store embeddings in Pinecone (`pip install pinecone-client`).  
   * Query Pinecone to retrieve relevant documents and feed them into the prompt.  
4. **Build the application** –  
   * Wrap the LangChain workflow in a web framework (FastAPI, Flask) or a chatbot UI.  
   * Deploy the LLM and Pinecone index to a cloud provider.  

Result: a modular LLM app that can reason, retrieve context, and act autonomously.
Number of Retrieved Context Documents: 4


In [7]:
# Step 7: Architecture Verification and Final Summary
print("================================================================================")
print("[VERIFICATION SUMMARY] Stage 05 - Lecture 42 Pipeline Validation")
print("================================================================================")
print("[PASS] 1. Raw corpus initialized (5 documents).")
print("[PASS] 2. FAISS dense retriever configured (all-MiniLM-L6-v2, k=3).")
print("[PASS] 3. BM25 sparse retriever configured (rank-bm25, k=3).")
print(f"[PASS] 4. EnsembleRetriever fused ranks with weights [{dense_weight}, {sparse_weight}].")
print("[PASS] 5. Irrelevant documents filtered out of top results.")
print("[PASS] 6. End-to-end RAG chain executed with Groq LLM synthesis.")
print("================================================================================")


[VERIFICATION SUMMARY] Stage 05 - Lecture 42 Pipeline Validation
[PASS] 1. Raw corpus initialized (5 documents).
[PASS] 2. FAISS dense retriever configured (all-MiniLM-L6-v2, k=3).
[PASS] 3. BM25 sparse retriever configured (rank-bm25, k=3).
[PASS] 4. EnsembleRetriever fused ranks with weights [0.7, 0.3].
[PASS] 5. Irrelevant documents filtered out of top results.
[PASS] 6. End-to-end RAG chain executed with Groq LLM synthesis.
